email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from dataclasses import dataclass

# define a dataclass `EmailContext` with defaults:
# email_address = "julie@example.com", password = "password123"
@dataclass
class EmailContext:
    email_address = "julie@example.com"
    password = "password123"

In [5]:
from langchain.agents import AgentState

# define `AuthenticatedState(AgentState)` with a bool field `authenticated`
class AuthenticatedState(AgentState):
    authenticated: bool

In [9]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

# write three @tool functions:
# - check_inbox(): return a hardcoded email from Jane (jane@example.com) asking Julie
#   to grab coffee next week
# - send_email(to, subject, body): return a string confirming the email was sent
# - authenticate(email, password, runtime: ToolRuntime) -> Command:
#   compare against runtime.context.email_address / .password; return a Command that
#   sets state["authenticated"] True/False AND adds a ToolMessage
#   (with tool_call_id=runtime.tool_call_id) saying success or failure
@tool
def check_inbox() -> str:
    """Check the inbox and return the emails found inside (hardcoded for now)"""
    return "from jane@example.com - would you like to grab coffee next week Julie?"

@tool
def send_email(to: str, subject: str, body:str) -> str:
    """Send an email to the recipient given both subject with a email body"""
    return f"Email sent - RE: {subject} to: {to} content: {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Check email and password credentials and authenticate/reject user as necessary based on credentials"""
    ctx_email_address, ctx_password = runtime.context.email_address, runtime.context.password
    authenticated = ctx_email_address == email and ctx_password == password
    return Command(update={
        "authenticated": authenticated, 
        "messages": [ToolMessage(f"Has authentication succeeded: {str(authenticated)}", tool_call_id=runtime.tool_call_id)]}
    )

In [12]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

# TODO: write a @wrap_model_call function `dynamic_tool_call(request, handler)`:
# - read request.state.get("authenticated")
# - if authenticated, expose [check_inbox, send_email]; otherwise only [authenticate]
# - request = request.override(tools=tools), then return handler(request)
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Dynamically call tools based on the request state"""
    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate] # non authenticated users must authenticate
    request = request.override(tools=tools) 

    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [13]:
from langchain.agents.middleware import dynamic_prompt

# TODO: define an authenticated prompt (assistant that checks inbox and sends emails;
# first step after auth is to check the inbox) and an unauthenticated prompt
# (assistant that can authenticate users)
# TODO: write a @dynamic_prompt function `dynamic_prompt(request)` that returns one
# or the other based on request.state.get("authenticated")
@dynamic_prompt
def dynamic_prompt_func(request: ModelRequest) -> str:
    """Generate system prompt based on authenticated."""
    authenticated = request.state.get("authenticated")
    authenticated_prompt = "You are a helpful email assistant. Check the inbox first, then help the user read and reply to emails."
    unauthenticated_prompt = "You are a helpful assistant. Ask the user for their email and password, then authenticate them. You cannot do anything else until they are authenticated."


    return authenticated_prompt if authenticated else unauthenticated_prompt

In [24]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

# create `agent` with create_agent:
# - model "gpt-5-nano", tools=[authenticate, check_inbox, send_email]
# - checkpointer=InMemorySaver() (needed so the interrupt can be resumed)
# - state_schema=AuthenticatedState, context_schema=EmailContext
# - middleware=[dynamic_tool_call, dynamic_prompt, HumanInTheLoopMiddleware(...)]
#   where interrupt_on requires approval ONLY for send_email
#   (authenticate: False, check_inbox: False, send_email: True)
agent = create_agent(
    model="gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[dynamic_tool_call, dynamic_prompt_func, HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
            },
            description_prefix="Tool execution requires approval for sending an email",
    )]
)

In [19]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

You have a new email from jane@example.com:

- Subject: (not shown)
- Body: would you like to grab coffee next week Julie?

Would you like me to reply? Here are a few quick options you could use. Tell me which one you prefer, or customize it:

Option 1 — simple and accommodating
Hi Jane, that sounds great! I’m free Tuesday and Thursday afternoon next week. Do either of those work for you?

Option 2 — ask for a specific time
Hi Jane, I’d love to! How about Tuesday at 3:00 PM or Thursday at 11:00 AM? Let me know what works.

Option 3 — flexible, with a suggestion
Hi Jane, yes please. I’m open early next week—perhaps Tuesday or Wednesday? If you have a preferred time, I can adjust.

If you want, I can compose and send one of these now. Which option or details would you like to use?


In [20]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [21]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane, that sounds great! I’m free Tuesday and Thursday afternoon next week. Do either of those work for you?


In [22]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Email sent to jane@example.com with the draft:
Subject: Re: Coffee next week
Body: Hi Jane, that sounds great! I’m free Tuesday and Thursday afternoon next week. Do either of those work for you?

Would you like me to draft a follow-up message if you don’t hear back in a couple of days, or tweak the reply in any way?


In [23]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='1ee47985-5bbf-427c-861d-ce30bbce5119'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 350, 'prompt_tokens': 176, 'total_tokens': 526, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EYPUFsINgvb5Z3ZbRX8bINUYvck1d', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a133c3-c398-7772-8ab3-c83254882004-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_h3p8JZ06pI3oR5blT